<a href="https://colab.research.google.com/github/abdrapsandani/DataScience_240401010174_AbdullahRapsandani/blob/main/Pertemuan12_AbdullahRapsandani_240401010174.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Nama Lengkap  : Abdullah Rapsandani
#### NIM           : 240401010174
#### Kelas         : IF403

# **Sesi 12 – Asosiasi Data & Sistem Rekomendasi Dasar**

#### STEP 1 — Generate & Eksplorasi Dataset Transaksi

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
filterwarnings('ignore')

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print("Contoh transaksi")
for i in range(5):
    print(f"Transaksi {i+1} :", transaksi[i])

print("Jumlah transaksi :", len(transaksi))

In [ ]:
# Mengubah Menjadi DataFrame

df_transaksi = pd.DataFrame({

    "ID_Transaksi": range(1,51),

    "Daftar_Item": [
        ", ".join(x)
        for x in transaksi
    ]
})

df_transaksi.head()

In [ ]:
from collections import Counter

# Menghitung Frekuensi Produk

counter = Counter()
for trx in transaksi:
    counter.update(trx)

frekuensi = pd.DataFrame(
    counter.items(),
    columns=[
        "Produk",
        "Frekuensi"
    ]
)

frekuensi = frekuensi.sort_values(
    by="Frekuensi",
    ascending=False
)

frekuensi

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(
    frekuensi["Produk"],
    frekuensi["Frekuensi"]
)

plt.title("Frekuensi Kemunculan Produk")
plt.xlabel("Produk")
plt.ylabel("Jumlah Kemunculan")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

In [ ]:
print(
    "Produk paling sering muncul:",
    frekuensi.iloc[0]["Produk"]
)

print(
    "Frekuensi:",
    frekuensi.iloc[0]["Frekuensi"]
)

#### STEP 2 — One-Hot Encoding Transaksi

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)

df = pd.DataFrame(te_ary, columns=te.columns_)

print(df.head())

#### STEP 3 — Cari Frequent Itemset dengan Apriori

In [ ]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(
        df,
        min_support=ms,
        use_colnames=True
    )

    print(f"min_support = {ms} : {len(freq)} itemset ditemukan")

In [ ]:
freq_items = apriori(
    df,
    min_support=0.1,
    use_colnames=True
)

freq_items = freq_items.sort_values(
    by="support",
    ascending=False
)

freq_items["Support (%)"] = (
    freq_items["support"] * 100
).round(2)

freq_items.head(10)

#### STEP 4 — Bentuk & Saring Aturan Asosiasi


In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.6
)

print("Jumlah aturan :", len(rules))
rules.head()

In [ ]:
rules = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

rules.head(10)

rules = rules.sort_values(
    by="confidence",
    ascending=False
)

rules.head(10)

rules["Support (%)"] = (
    rules["support"]*100
).round(2)

rules["Confidence (%)"] = (
    rules["confidence"]*100
).round(2)

rules.head(10)

best_rule = rules.iloc[0]

print("Antecedent :", best_rule["antecedents"])
print("Consequent :", best_rule["consequents"])
print("Support :", round(best_rule["support"],3))
print("Confidence :", round(best_rule["confidence"],3))
print("Lift :", round(best_rule["lift"],3))

In [ ]:
for i, row in rules.iterrows():
    print("="*50)
    print(f"Rule {i+1}")
    print(f"{set(row['antecedents'])} --> {set(row['consequents'])}")
    print(f"Support    : {row['support']:.2f}")
    print(f"Confidence : {row['confidence']:.2f}")
    print(f"Lift       : {row['lift']:.2f}")

In [ ]:
top_rules = rules.head(10)

plt.figure(figsize=(10,5))

plt.bar(
    range(len(top_rules)),
    top_rules["confidence"]
)

plt.xticks(
    range(len(top_rules)),
    [
        f"{list(a)[0]}→{list(c)[0]}"
        for a,c in zip(
            top_rules["antecedents"],
            top_rules["consequents"]
        )
    ],
    rotation=45
)

plt.ylabel("Confidence")
plt.title("Top Association Rules")
plt.grid(axis="y")
plt.show()

In [ ]:
rules.sort_values(by="lift", ascending=False).head(10)

#### STEP 5 — Rekomender Sederhana dengan Content-Based Filtering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

produk = pd.DataFrame({
    "Produk": [
        "Roti",
        "Selai",
        "Susu",
        "Keju",
        "Mentega",
        "Kopi",
        "Teh",
        "Sereal",
        "Telur",
        "Gula"
    ],
    "Kategori": [
        "Makanan Sarapan Roti",
        "Makanan Olesan Sarapan",
        "Minuman Susu",
        "Produk Olahan Susu",
        "Makanan Olesan Roti",
        "Minuman Panas",
        "Minuman Panas",
        "Makanan Sarapan",
        "Protein Sarapan",
        "Pemanis Minuman"
    ]
})

produk

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(
    produk["Kategori"]
)

cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim
similarity_df = pd.DataFrame(
    cosine_sim,
    index=produk["Produk"],
    columns=produk["Produk"]
)

similarity_df
def rekomendasi(nama_produk, top_n=3):
    skor = similarity_df[nama_produk]
    rekom = skor.sort_values(
        ascending=False
    )[1:top_n+1]
    return rekom

hasil = rekomendasi("Roti").reset_index()
hasil.columns = [
    "Produk Rekomendasi",
    "Similarity"
]
hasil

#### STEP 6 — Bandingkan Kedua Pendekatan

In [ ]:
print("=== Association Rule ===")
print("Jika pelanggan membeli: Roti")
print("Maka direkomendasikan: Selai")
print("Confidence : 68.75%")
print("Lift : 1.32")

print("\n")

print("=== Content-Based Filtering ===")
hasil = rekomendasi("Roti")
print(hasil)

In [ ]:
perbandingan = pd.DataFrame({
    "Aspek": [
        "Dasar Rekomendasi",
        "Sumber Data",
        "Kelebihan",
        "Kekurangan"
    ],
    "Association Rule": [
        "Pola pembelian bersama",
        "Riwayat transaksi",
        "Menemukan kebiasaan pelanggan",
        "Perlu banyak data transaksi"
    ],
    "Content-Based": [
        "Kemiripan karakteristik produk",
        "Informasi atribut produk",
        "Tidak membutuhkan riwayat transaksi pengguna lain",
        "Bergantung pada kualitas atribut produk"
    ]
})

perbandingan

# Kesimpulan

Pada praktikum ini dipelajari penerapan Market Basket Analysis menggunakan algoritma Aprioriuntuk menemukan pola pembelian produk yang sering muncul bersama. Selain itu, dipelajari cara menggunakan Support, Confidence, dan Lift untuk menilai kekuatan aturan asosiasi. Praktikum juga memperkenalkan Content-Based Filtering dengan menggunakan kemiripan kategori produk untuk menghasilkan rekomendasi.

Temuan utama menunjukkan bahwa aturan asosiasi dapat digunakan untuk mengetahui produk yang cenderung dibeli secara bersamaan, sedangkan Content-Based Filtering dapat memberikan rekomendasi berdasarkan kemiripan atribut produk. Kedua pendekatan kemudian dapat dibandingkan untuk melihat apakah rekomendasi yang dihasilkan konsisten dan menentukan pendekatan yang lebih sesuai dengan kebutuhan.
Keterbatasan praktikum ini adalah dataset yang digunakan merupakan data sintetis dengan 50 transaksi dan 10 produk, sehingga hasilnya belum tentu menggambarkan pola pembelian pelanggan pada kondisi nyata. Selain itu, Content-Based Filtering pada praktikum ini hanya menggunakan kategori produk sebagai fitur. Pengembangan selanjutnya dapat menggunakan data transaksi yang lebih besar, atribut produk yang lebih lengkap, atau menggabungkan Association Rules dan Content-Based Filtering menjadi pendekatan hybrid.
